# Day 4 模块 6：怎么选模型 + 今日收束

把评估、对比、调参、残差收成一张决策小抄，并固定**本项目默认模型**。


## 1. 选模型时看什么（高中版）

| 问题 | 更在乎时 |
| --- | --- |
| 分数够不够 | 测试 R² / MAE |
| 会不会背题 | 训练分 vs 测试分 |
| 好不好讲 | 线性系数、单棵树、特征重要性 |
| 好不好维护 | 依赖少、调参别太玄 |
| 能不能帮决策 | 重要性 + 情景推演能不能说清 |

### 本项目默认结论

```text
主模型：随机森林（RandomForestRegressor）
原因：测试表现稳 + 特征重要性好讲 + 后面做「假如…会怎样」方便
线性回归：可作对照基线（简单、快）
单棵深树：演示过拟合
KNN：演示「不是所有算法都适合这份表」
```


## 2. 和 Day 5 的衔接（结营主线）

明天不是再学一个新算法，而是换帽子：

```text
Day 4 你刚会的
  评估指标 · 多模型对比 · 调参感觉 · 残差 · 默认 RF
        ↓
Day 5 要用的
  经营场景对比 · 重要性说成业务话 · what-if 推演
  → 一页经营诊断 → 结营路演
```

可选：把预测挂到网页（`mini_predict_app.py`），**主课时间给经营分析与结营**。

所以今天请确认你能独立完成（明天直接复用）：

1. 准备 `X, y` 并划分
2. 训 RF 并算测试 R² / MAE
3. 读 `feature_importances_`，能说出 Top 几名
4. 对一行新特征 `predict`（后面 what-if 就靠这个）


In [ ]:
# 最小闭环自测（应能一次跑通）
from pathlib import Path
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error

df = pd.read_csv('day4_cafe_sales.csv')
df['weather_score'] = df['weather_label'].map(
    {'晴': 1.0, '多云': 0.8, '阴': 0.6, '小雨': 0.4, '大雨': 0.3}
)
X = df[['price', 'promotion', 'is_weekend', 'temp', 'quality',
        'competitors', 'holiday', 'location', 'weather_score']]
y = df['sales']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
rf = RandomForestRegressor(n_estimators=100, max_depth=8, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
pred = rf.predict(X_test)
print('测试 R²:', round(r2_score(y_test, pred), 3))
print('测试 MAE:', round(mean_absolute_error(y_test, pred), 1))
print('Top3 特征:')
imp = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)
print(imp.head(3).round(3))


## 3. 今日你应能说清

1. R² / MAE / RMSE 各是什么感觉
2. 为什么要多模型对比，而不能只报一个训练分
3. 超参数、交叉验证、网格搜索各自解决什么问题
4. 残差如何帮你找「猜得差的天」
5. 本项目为什么默认随机森林
6. （衔接）指标和重要性，明天怎么改成「给店长听的话」
